# Sentiment Classification of Bitcoin Tweets: From Dictionaries to Transformers

*Based on the second assignment of the course "Deep Learning in Text Analytics"
(Prof. Stefan Lessmann, Humboldt-Universität zu Berlin). Reorganized and revised.*

**Question.** How well do different text-analysis methods measure sentiment in
short financial social media posts, and how much do modern deep learning models
improve on the word-list (dictionary) approach that is common in accounting and
finance research?

**Approach.** Six classifiers are compared on the same train/test split:

| # | Model | Idea |
|---|---|---|
| 1 | Dictionary (Harvard IV-4) | Count positive vs. negative words, no training |
| 2 | TF-IDF + XGBoost | Bag-of-words features + gradient boosted trees |
| 3 | Bidirectional GRU | Recurrent network, embeddings learned from scratch |
| 4 | Bidirectional LSTM + Word2Vec | Recurrent network, pretrained word embeddings |
| 5 | DistilBERT (zero-shot) | Transformer fine-tuned on movie reviews, applied as is |
| 6 | DistilBERT (fine-tuned) | Transformer fine-tuned on our tweets |

**Structure.** This notebook contains the analysis steps and results. The
reusable code (text cleaning, model builders, evaluation) lives in `src/`:

| File | Content |
|---|---|
| `src/preprocessing.py` | Tweet cleaning |
| `src/baselines.py` | Dictionary score, TF-IDF + XGBoost |
| `src/rnn.py` | Vocabulary, embeddings, GRU/LSTM builder |
| `src/bert.py` | DistilBERT zero-shot and fine-tuning |
| `src/evaluation.py` | Metrics and plots shared by all models |

## 1. Setup

In [ ]:
from pathlib import Path

# src.bert (Hugging Face / PyTorch) is imported before Keras / TensorFlow:
# importing transformers after TensorFlow can crash the kernel on some systems.
from src.bert import fine_tune_distilbert, pipeline_positive_probability, predict_positive_probability

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors
from sklearn.utils import class_weight

from src.preprocessing import TextPreprocessor, download_nltk_resources, light_clean
from src.baselines import dictionary_polarity, train_tfidf_xgboost
from src.rnn import (
    build_bidirectional_rnn_classifier,
    build_embedding_matrix,
    build_vocabulary,
    texts_to_padded_sequences,
)
from src.evaluation import METRIC_NAMES, evaluate_binary_classifier, plot_roc_curves, plot_training_history

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and the backend
download_nltk_resources()

DATA_DIR = Path("data")
W2V_PATH = DATA_DIR / "w2v_imdb_full_d100_e500.model"  # see README for download link

results = {}  # model name -> dict of metrics
scores = {}   # model name -> test-set scores, used for ROC curves

## 2. Data

In [ ]:
df_train = pd.read_parquet(DATA_DIR / "btc_tweets_train.parquet.gzip")
df_test = pd.read_parquet(DATA_DIR / "btc_tweets_test.parquet.gzip")
for df in (df_train, df_test):
    df["sentiment"] = df["sentiment"].astype(int)

print(f"Train: {len(df_train)} tweets, test: {len(df_test)} tweets")
df_train[["content", "sentiment"]].head()

In [ ]:
balance = pd.DataFrame({
    "train": df_train["sentiment"].value_counts(normalize=True),
    "test": df_test["sentiment"].value_counts(normalize=True),
}).rename(index={1: "positive", 0: "negative"})
balance.style.format("{:.1%}")

The classes are imbalanced: about 81% of tweets are positive. A model that always
predicts "positive" would already reach about 81% accuracy, so **AUC and F1 are
more informative than accuracy** here. The imbalance is handled with class
weights (RNNs, XGBoost) or by downsampling (DistilBERT fine-tuning).

In [ ]:
majority_accuracy = (df_test["sentiment"] == 1).mean()
print(f"Accuracy of always predicting 'positive' on the test set: {majority_accuracy:.3f}")

y_train = df_train["sentiment"].to_numpy()
y_test = df_test["sentiment"].to_numpy()
weights = class_weight.compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}
print("Class weights:", class_weights)

## 3. Text preprocessing

Two versions of each tweet are created:

* **`clean_content`** for the dictionary, TF-IDF and RNN models: lowercased, with
  URLs, mentions, emojis, punctuation and stopwords removed, then lemmatized.
  Negation words (*no, not, nor*) are **kept**, since removing them can flip the
  sentiment ("not good" → "good").
* **`light_content`** for the DistilBERT models: only URLs and mentions removed.
  Transformers are pretrained on natural text and use casing, punctuation and
  function words as signals.

In [ ]:
cleaner = TextPreprocessor()
for df in (df_train, df_test):
    df["clean_content"] = df["content"].apply(cleaner.clean)
    df["light_content"] = df["content"].apply(light_clean)

pd.set_option("display.max_colwidth", 120)
df_test[["content", "clean_content"]].head(5)

## 4. Models

All models are trained on the training set only and evaluated on the same
held-out test set of 500 tweets.

### 4.1 Dictionary approach (Harvard IV-4)

Polarity = (positive − negative words) / (positive + negative words). A tweet is
classified as positive if its polarity is above 0. This mirrors the word-list
approach widely used to measure tone in financial text.

In [ ]:
scores["Dictionary (HIV4)"] = dictionary_polarity(df_test["clean_content"])
results["Dictionary (HIV4)"] = evaluate_binary_classifier(
    y_test, scores["Dictionary (HIV4)"], cutoff=0, model_name="Dictionary (HIV4)"
)

### 4.2 TF-IDF + XGBoost

Each tweet is represented by TF-IDF weights of its words; an XGBoost classifier
learns which words predict sentiment.

In [ ]:
tfidf_xgb = train_tfidf_xgboost(df_train["clean_content"], y_train, seed=SEED)
scores["TF-IDF + XGBoost"] = tfidf_xgb.predict_proba(df_test["clean_content"])[:, 1]
results["TF-IDF + XGBoost"] = evaluate_binary_classifier(
    y_test, scores["TF-IDF + XGBoost"], model_name="TF-IDF + XGBoost"
)

### 4.3 Bidirectional GRU (embeddings learned from scratch)

Tweets are converted into sequences of word ids (vocabulary of the 4,000 most
frequent training words) and padded to equal length.

In [ ]:
NUM_WORDS = 4000
vocabulary = build_vocabulary(df_train["clean_content"], NUM_WORDS)
MAX_LEN = max(len(text.split()) for text in df_train["clean_content"])
print(f"Longest training tweet: {MAX_LEN} tokens")

X_train_seq = texts_to_padded_sequences(df_train["clean_content"], vocabulary, MAX_LEN)
X_test_seq = texts_to_padded_sequences(df_test["clean_content"], vocabulary, MAX_LEN)

RNN_PARAMS = dict(hidden_units=16, learning_rate=0.001)
FIT_PARAMS = dict(epochs=5, batch_size=64, validation_split=0.25,
                  class_weight=class_weights, verbose=0)

In [ ]:
bi_gru = build_bidirectional_rnn_classifier(
    vocab_size=NUM_WORDS, embedding_dim=50, rnn_type="gru", **RNN_PARAMS
)
history_gru = bi_gru.fit(X_train_seq, y_train, **FIT_PARAMS)

scores["BiGRU"] = bi_gru.predict(X_test_seq, verbose=0).ravel()
results["BiGRU"] = evaluate_binary_classifier(y_test, scores["BiGRU"], model_name="BiGRU")

### 4.4 Bidirectional LSTM with pretrained Word2Vec embeddings

The embedding layer starts from 100-dimensional Word2Vec vectors trained on IMDB
movie reviews and is further trained on the tweets.

In [ ]:
word_vectors = KeyedVectors.load_word2vec_format(W2V_PATH, binary=False)
w2v_matrix, _ = build_embedding_matrix(vocabulary, word_vectors, NUM_WORDS)

bi_lstm = build_bidirectional_rnn_classifier(
    vocab_size=NUM_WORDS, embedding_dim=w2v_matrix.shape[1], rnn_type="lstm",
    embedding_weights=w2v_matrix, trainable_embedding=True, **RNN_PARAMS
)
history_lstm = bi_lstm.fit(X_train_seq, y_train, **FIT_PARAMS)

scores["BiLSTM + Word2Vec"] = bi_lstm.predict(X_test_seq, verbose=0).ravel()
results["BiLSTM + Word2Vec"] = evaluate_binary_classifier(
    y_test, scores["BiLSTM + Word2Vec"], model_name="BiLSTM + Word2Vec"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_training_history(history_gru, "BiGRU", ax=axes[0])
plot_training_history(history_lstm, "BiLSTM + Word2Vec", ax=axes[1])
plt.tight_layout()
plt.show()

### 4.5 DistilBERT, zero-shot

A DistilBERT model already fine-tuned on movie-review sentiment (SST-2) is
applied directly to the tweets, without any training on our data.

In [ ]:
scores["DistilBERT zero-shot"] = pipeline_positive_probability(df_test["light_content"])
results["DistilBERT zero-shot"] = evaluate_binary_classifier(
    y_test, scores["DistilBERT zero-shot"], model_name="DistilBERT zero-shot"
)

### 4.6 DistilBERT, fine-tuned on the tweets

The training set is balanced by randomly downsampling positive tweets to the
number of negative tweets, then DistilBERT is fine-tuned for 2 epochs.

In [ ]:
df_neg = df_train[df_train["sentiment"] == 0]
df_pos = df_train[df_train["sentiment"] == 1].sample(n=len(df_neg), random_state=SEED)
df_balanced = pd.concat([df_neg, df_pos]).sample(frac=1, random_state=SEED)
print(f"Balanced training set: {len(df_balanced)} tweets")

trainer, tokenizer = fine_tune_distilbert(
    df_balanced["light_content"], df_balanced["sentiment"], num_train_epochs=2, seed=SEED
)
scores["DistilBERT fine-tuned"] = predict_positive_probability(trainer, tokenizer, df_test["light_content"])
results["DistilBERT fine-tuned"] = evaluate_binary_classifier(
    y_test, scores["DistilBERT fine-tuned"], model_name="DistilBERT fine-tuned"
)

## 5. Results

In [ ]:
results_table = pd.DataFrame({name: {m: r[m] for m in METRIC_NAMES} for name, r in results.items()}).T
results_table.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold")

In [ ]:
plot_roc_curves(y_test, scores)
plt.show()

## 6. Discussion and limitations

**Findings.**
* **The dictionary approach performs worst** (AUC ≈ 0.65). Many tweets contain
  no Harvard IV-4 words at all, and some general-language words carry a different
  meaning in crypto discussions. This mirrors the finding of Loughran and McDonald
  (2011) that general-purpose word lists misclassify tone in financial text.
* **Learning from labeled data helps substantially.** TF-IDF + XGBoost, a simple
  bag-of-words model, reaches AUC ≈ 0.78.
* **Accuracy alone is misleading with imbalanced classes.** A model can match the
  81% accuracy of always predicting "positive" while recognizing very few negative
  tweets, which is why the confusion matrices and AUC are reported for every model.
* [Complete after running the notebook: how the two DistilBERT models compare
  with the other approaches.]

**Limitations.**
* **Small sample.** 1,500 training and 500 test tweets; differences of a few
  percentage points between models may not be robust to another split.
* **Domain mismatch.** The Harvard IV-4 dictionary, the Word2Vec vectors (IMDB)
  and the zero-shot DistilBERT (SST-2) were all built for general or movie-review
  language, not crypto slang ("moon", "HODL", "rekt"). Loughran and McDonald
  (2011) show the same problem for financial text, where general-purpose word
  lists misclassify many words; a finance-specific dictionary or a model such as
  FinBERT would be the natural next step.
* **Label quality.** The sentiment labels come with the course dataset and their
  construction is not documented, which limits how far the results generalize.
* **Single run.** Neural network results vary with the random seed; repeated
  runs or cross-validation would give more reliable comparisons.